In [28]:
import pandas as pd
import numpy as np
import matplotlib as plt
import ast
from collections import Counter


import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
pd.set_option('display.max_columns',100)          
pd.set_option('display.max_rows',100) 
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error





In [30]:
df = pd.read_csv("clean_data/clean_paris_kaggle.csv", encoding='ISO-8859-1',low_memory=False)

df.columns
###feature selection and pre processing

df = df.drop(columns=["host_since","room_type"])

bool_cols = df.select_dtypes(include="bool").columns
df.isna().sum()



host_is_superhost                 0
latitude                          0
longitude                         0
host_total_listings_count         0
host_has_profile_pic              0
host_identity_verified            0
accommodates                      0
bedrooms                          0
price                             0
minimum_nights                    0
maximum_nights                    0
review_scores_rating              0
review_scores_cleanliness         0
review_scores_checkin             0
review_scores_communication       0
review_scores_location            0
review_scores_value               0
instant_bookable                  0
Has_private_entrance              0
Has_crib                          0
Has_keypad                        0
Has_freezer                       0
Has_cooking_basics                0
Has_dryer                         0
Has_stove                         0
Has_lock_on_bedroom_door          0
Has_heating                       0
Has_hot_water               

In [31]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)



(30384, 88) (3798, 88) (3798, 88)


In [32]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)


In [33]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns

num_cols, cat_cols


(Index(['host_is_superhost', 'latitude', 'longitude',
        'host_total_listings_count', 'host_has_profile_pic',
        'host_identity_verified', 'accommodates', 'bedrooms', 'minimum_nights',
        'maximum_nights', 'review_scores_rating', 'review_scores_cleanliness',
        'review_scores_checkin', 'review_scores_communication',
        'review_scores_location', 'review_scores_value', 'instant_bookable',
        'Has_private_entrance', 'Has_crib', 'Has_keypad', 'Has_freezer',
        'Has_cooking_basics', 'Has_dryer', 'Has_stove',
        'Has_lock_on_bedroom_door', 'Has_heating', 'Has_hot_water',
        'Has_babysitter_recommendations', 'Has_coffee_maker', 'Has_essentials',
        'Has_kitchen', 'Has_baby_bath', 'Has_elevator', 'Has_baking_sheet',
        'Has_body_soap', 'Has_bathtub', 'Has_dishwasher', 'Has_shampoo',
        'Has_paid_parking_off_premises', 'Has_carbon_monoxide_alarm',
        'Has_dishes_and_silverware', 'Has_smoke_alarm', 'Has_bread_maker',
        'Has_c

In [34]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])



In [35]:
def evaluate(model, X, y, name):
    preds = model.predict(X)
    print(name)
    print("MAE:", mean_absolute_error(y, preds))
    print("RMSE:", root_mean_squared_error(y, preds))
    print("R2:", r2_score(y, preds))
    print("-" * 30)


In [36]:
ols = Pipeline([
    ("prep", preprocessor),
    ("model", LinearRegression())
])

ols.fit(X_train, y_train)
evaluate(ols, X_val, y_val, "OLS")


OLS
MAE: 38.77099308186518
RMSE: 61.12511904312543
R2: 0.4484556236837034
------------------------------


In [37]:
ridge = Pipeline([
    ("prep", preprocessor),
    ("model", Ridge(alpha=1.0))
])

lasso = Pipeline([
    ("prep", preprocessor),
    ("model", Lasso(alpha=0.01))
])

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

evaluate(ridge, X_val, y_val, "Ridge")
evaluate(lasso, X_val, y_val, "Lasso")


Ridge
MAE: 38.77079930021214
RMSE: 61.12505161062612
R2: 0.4484568405973338
------------------------------
Lasso
MAE: 38.75672351889966
RMSE: 61.12120343275806
R2: 0.4485262841172496
------------------------------


In [38]:
rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
evaluate(rf, X_val, y_val, "Random Forest")


Random Forest
MAE: 33.46332316507435
RMSE: 54.848308117830285
R2: 0.5559135765881581
------------------------------


In [39]:
gbr = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

gbr.fit(X_train, y_train)
evaluate(gbr, X_val, y_val, "Gradient Boosting")


Gradient Boosting
MAE: 33.853743406895425
RMSE: 55.886785328799135
R2: 0.538938049345355
------------------------------


In [40]:
mlp = Pipeline([
    ("prep", preprocessor),
    ("model", MLPRegressor(
        hidden_layer_sizes=(128, 64),
        max_iter=300,
        random_state=42
    ))
])

mlp.fit(X_train, y_train)
evaluate(mlp, X_val, y_val, "Neural Network")


Neural Network
MAE: 48.171141872184805
RMSE: 72.32631383211097
R2: 0.22779308224178074
------------------------------


/home/alant/.local/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [48]:
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

def accuracy_within_ranges(y_true, y_pred, euro_ranges, pct_ranges):
    abs_err = np.abs(y_true - y_pred)
    pct_err = abs_err / y_true

    abs_acc = {
        f"±€{r}": np.mean(abs_err <= r) * 100
        for r in euro_ranges
    }
    pct_acc = {
        f"±{int(p*100)}%": np.mean(pct_err <= p) * 100
        for p in pct_ranges
    }
    return abs_acc, pct_acc


In [45]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
    oob_score=True
)

param_grid = {
    "n_estimators": [200],
    "max_depth": [15, 20, None],
    "min_samples_split": [5, 10],
    "min_samples_leaf": [3, 5]
}

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
grid.best_params_


Fitting 5 folds for each of 12 candidates, totalling 60 fits


{'max_depth': 20,
 'min_samples_leaf': 3,
 'min_samples_split': 5,
 'n_estimators': 200}

In [46]:
from sklearn.model_selection import train_test_split

X_trainval = np.vstack([X_train, X_val])
y_trainval = np.hstack([y_train, y_val])

best_rf.fit(X_trainval, y_trainval)


,n_estimators,200
,criterion,'squared_error'
,max_depth,20
,min_samples_split,5
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


In [49]:
y_pred_test = best_rf.predict(X_test)

metrics_test = regression_metrics(y_test, y_pred_test)
metrics_test


/home/alant/.local/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


{'MAE': 33.361075414807175, 'RMSE': 56.8169846651567, 'R2': 0.5237169529053128}

In [50]:
best_rf.oob_score_


0.5813068932821488

In [51]:
euro_ranges = [100, 200, 500, 1000, 2000]
pct_ranges = [0.10, 0.20, 0.30, 0.50]

abs_acc, pct_acc = accuracy_within_ranges(
    y_test, y_pred_test,
    euro_ranges, pct_ranges
)

abs_acc, pct_acc


({'±€100': np.float64(94.15481832543445),
  '±€200': np.float64(98.76250658241179),
  '±€500': np.float64(99.97367035281727),
  '±€1000': np.float64(100.0),
  '±€2000': np.float64(100.0)},
 {'±10%': np.float64(22.880463401790415),
  '±20%': np.float64(42.96998420221169),
  '±30%': np.float64(59.347024749868346),
  '±50%': np.float64(78.67298578199052)})

In [52]:
import pandas as pd

importances = pd.Series(
    best_rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importances.head(15)


bedrooms                     0.357035
latitude                     0.091164
longitude                    0.084642
host_total_listings_count    0.078203
accommodates                 0.047050
review_scores_rating         0.027714
Has_air_conditioning         0.025849
minimum_nights               0.018028
maximum_nights               0.015728
Has_dishwasher               0.012742
Has_tv                       0.012171
Has_dryer                    0.012028
review_scores_cleanliness    0.011532
Has_high_chair               0.007886
review_scores_value          0.007560
dtype: float64